<a href="https://colab.research.google.com/github/peremartra/Rearchitecting-LLMs/blob/main/CH08/CH08_NB01_KVCache_HuggingFace.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<table style="width:100%; border:none; background:none;">
  <tr style="border:none;">
    <td style="border:none; vertical-align:middle; text-align:left; width: 120px;">
      <a href="https://hubs.la/Q040tvsK0"><img src="https://raw.githubusercontent.com/peremartra/Rearchitecting-LLMs/main/Images/cover.png" width="100px" style="border-radius: 4px;"></a>
    </td>
    <td style="border:none; vertical-align:middle; text-align:left;">
      <p style="margin: 0; font-size: 14px;">
        Supplementary code for the <a href="https://hubs.la/Q040tvsK0">Rearchitecting LLMs</a> book by <a href="https://github.com/peremartra">Pere Martra</a>.<br>
        <br>
        Code repository: <a href="https://github.com/peremartra/Rearchitecting-LLMs">https://github.com/peremartra/Rearchitecting-LLMs</a>
      </p>
    </td>
  </tr>
</table>

# Quantizing the KV Cache: Hugging Face
## Reducing the dynamic memory footprint during inference

[![LinkedIn](https://img.shields.io/badge/LinkedIn-0077B5?style=flat&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/pere-martra/) [![GitHub](https://img.shields.io/badge/GitHub-100000?style=flat&logo=github&logoColor=white)](https://github.com/peremartra) [![X](https://img.shields.io/badge/X-000000?style=flat&logo=x&logoColor=white)](https://x.com/PereMartra) [![Hugging Face](https://img.shields.io/badge/🤗%20Hugging%20Face-blue)](https://huggingface.co/oopere)

_____
* **Author:** Pere Martra
* **Models:** meta-llama/Llama-3.2-3B
* **Colab Environment:** T4 GPU (Free Tier)
* **Keys:**
  * KV Cache
  * Quantization
  * BitsAndBytes
  * Quanto
* **References:**
  * [Manning Book: Rearchitecting LLMs](https://livebook.manning.com/book/rearchitecting-llms)
  * [Quanto Documentation](https://github.com/huggingface/optimum-quanto)
_____

This notebook compares three inference configurations to isolate the effect of KV cache quantization on VRAM usage:

1. **FP16 baseline** — model and KV cache both in FP16.
2. **4-bit weight quantization (BitsAndBytes)** — smaller static footprint, but KV cache still grows in FP16.
3. **FP16 model + 4-bit KV cache (Quanto)** — static footprint unchanged, dynamic delta reduced.

By fixing one variable at a time we can clearly see which lever — weight precision or cache precision — controls which component of the total VRAM budget.

## 1. Install dependencies

In [ ]:
!pip install -q transformers==4.51.3
!pip install -q accelerate
!pip install -q bitsandbytes
!pip install -q optimum-quanto

## 2. Imports and configuration

In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    QuantizedCacheConfig,
)
import sys
sys.path.append('..')

!wget -q https://raw.githubusercontent.com/peremartra/Rearchitecting-LLMs/main/utils.py

from utils import get_output, measure_memory_allocation

MODEL_NAME = "meta-llama/Llama-3.2-3B"

# Long prompt to make KV cache growth visible in memory measurements
PROMPT = (
    "Explain in detail the history of artificial intelligence, "
    "covering all major milestones from the 1950s to the present day."
)
MAX_NEW_TOKENS = 500

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 3. Load tokenizer

We load the tokenizer once and reuse it across all three tests.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

## 4. Test 1 — Baseline: FP16

We start with the standard FP16 load. This establishes our ceiling:
- **Static VRAM** is high because all model weights live in FP16.
- **Dynamic delta** grows proportionally with `MAX_NEW_TOKENS` because the KV cache is also FP16.

All subsequent tests will be compared against these numbers.

In [ ]:
model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

results_fp16 = measure_memory_allocation(
    model_fp16, tokenizer, PROMPT,
    max_new_tokens=MAX_NEW_TOKENS
)
print(results_fp16)

del model_fp16
torch.cuda.empty_cache()

**Expected output:** Static VRAM ~6 GB, dynamic delta proportional to `MAX_NEW_TOKENS`.

Note that quantizing model weights (next test) will shrink the static footprint but leave the KV cache allocation unchanged.

## 5. Test 2 — Full model quantization: 4-bit weights (BitsAndBytes)

Here we quantize the model weights to 4-bit using BitsAndBytes. This dramatically reduces the **static** VRAM footprint.

The key insight is that BitsAndBytes quantizes *weights*, not the KV cache. During generation, the attention mechanism still materializes keys and values in FP16, so the **dynamic delta** should be nearly identical to Test 1.

This test isolates the weight-precision lever.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,  # Explicit to ensure comparable
)                                           # measurements across tests

model_4bit = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

results_4bit = measure_memory_allocation(
    model_4bit, tokenizer, PROMPT,
    max_new_tokens=MAX_NEW_TOKENS
)
print(results_4bit)

del model_4bit
torch.cuda.empty_cache()

**Expected output:** Static VRAM ~2–3 GB. Dynamic delta practically identical to Test 1 — the KV cache is still materialized in FP16.

## 6. Test 3 — KV cache quantization: 4-bit cache (Quanto)

Now we flip the lever: the model stays in FP16 (same static footprint as Test 1), but we inject a `QuantizedCacheConfig` that tells `model.generate()` to store keys and values in 4-bit via the Quanto backend.

The static VRAM should match Test 1. The dynamic delta should be significantly smaller because each KV entry now occupies 4 bits instead of 16.

A possible side-effect is a drop in throughput: quantizing and dequantizing the cache on every attention step adds CPU/GPU overhead, especially with the Python-level Quanto backend.

In [ ]:
model_kvcache = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

cache_config = QuantizedCacheConfig(
    backend="quanto",
    nbits=4
)

results_kvcache = measure_memory_allocation(
    model_kvcache, tokenizer, PROMPT,
    max_new_tokens=MAX_NEW_TOKENS,
    cache_config=cache_config        # Injected into generate()
)
print(results_kvcache)

del model_kvcache
torch.cuda.empty_cache()

**Expected output:** Static VRAM similar to Test 1. Dynamic delta significantly smaller. Throughput may decrease due to the quantization overhead applied at every generation step — this is expected and documented in the summary table below.

## 7. Results summary

The table below consolidates all three tests and makes the trade-offs explicit:

| Lever | Affects static VRAM? | Affects dynamic delta? |
|---|---|---|
| Weight quantization (BnB 4-bit) | ✅ Yes — large reduction | ❌ No |
| KV cache quantization (Quanto 4-bit) | ❌ No | ✅ Yes — large reduction |

In [ ]:
import pandas as pd

summary = pd.DataFrame([
    {
        "Configuration":           "FP16 Base",
        "Static VRAM (MB)":        results_fp16["static_vram_mb"],
        "Dynamic VRAM Delta (MB)": results_fp16["dynamic_delta_mb"],
        "Throughput (Tokens/s)":   results_fp16["throughput_tokens_s"],
    },
    {
        "Configuration":           "4-bit Weights (BnB)",
        "Static VRAM (MB)":        results_4bit["static_vram_mb"],
        "Dynamic VRAM Delta (MB)": results_4bit["dynamic_delta_mb"],
        "Throughput (Tokens/s)":   results_4bit["throughput_tokens_s"],
    },
    {
        "Configuration":           "FP16 + 4-bit KV Cache",
        "Static VRAM (MB)":        results_kvcache["static_vram_mb"],
        "Dynamic VRAM Delta (MB)": results_kvcache["dynamic_delta_mb"],
        "Throughput (Tokens/s)":   results_kvcache["throughput_tokens_s"],
    },
])

print(summary.to_string(index=False))